# End-to-end pipeline: copy baseline

In [1]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src" / "preprocessing" / "align.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the project checkout.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

## 1. Data

In [2]:
from src.data import TedData

ted = TedData()

## 2. Preprocessing

In [3]:
SRC_LANG, TGT_LANG = "en", "az"
DIRECTION = f"{SRC_LANG}-{TGT_LANG}"

train_pairs = [(s, t) for _, s, t in ted.pairs("train", SRC_LANG, TGT_LANG)][:1000]
dev_pairs = [(s, t) for _, s, t in ted.pairs("dev", SRC_LANG, TGT_LANG)][:100]
test_pairs = [(s, t) for _, s, t in ted.pairs("test", SRC_LANG, TGT_LANG)][:100]
len(train_pairs), len(dev_pairs), len(test_pairs)

(1000, 100, 100)

## 3. Model

In [4]:
from src.model import build_model, translate_corpus, save_translations

MODEL_NAME, SYSTEM, MAX_LENGTH = "copy", "copy", 20

model = build_model(MODEL_NAME, None, None)
model.fit(pairs=list(train_pairs))

## 4. Evaluation

In [5]:
from src.evaluation import evaluate_direction

hyps = translate_corpus(model, [s for s, _ in test_pairs], max_length=MAX_LENGTH)
out_dir = ROOT / "results" / "translations" / SYSTEM
save_translations(out_dir, DIRECTION, [s for s, _ in test_pairs], [t for _, t in test_pairs], hyps)

scores = {}
for metric in ("bleu", "chrf", "comet"):
    try:
        scores[metric] = evaluate_direction(out_dir, DIRECTION, metric)
    except Exception as e:
        scores[metric] = str(e)
scores

  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [00:00<00:00, 588261.43it/s]

/Users/kknopp/Documents/MNLP/code/.venv/lib/python3.12/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.6. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`


...

Predicting DataLoader 0:  85%|████████▍ | 11/13 [00:04<00:00,  2.63it/s]

Predicting DataLoader 0:  92%|█████████▏| 12/13 [00:04<00:00,  2.42it/s]

Predicting DataLoader 0: 100%|██████████| 13/13 [00:05<00:00,  2.38it/s]

Predicting DataLoader 0: 100%|██████████| 13/13 [00:05<00:00,  2.38it/s]

{'bleu': {'direction': 'en-az',
  'metric': 'bleu',
  'n_sentences': 100,
  'score': 1.0683288937260236,
  'signature': 'nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0'},
 'chrf': {'direction': 'en-az',
  'metric': 'chrf',
  'n_sentences': 100,
  'score': 10.956360116713634,
  'signature': 'nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no|version:2.6.0'},
 'comet': {'direction': 'en-az',
  'metric': 'comet',
  'n_sentences': 100,
  'score': 0.4738187685608864,
  'sentence_scores': [0.5727169513702393,
   0.6851343512535095,
   0.34300997853279114,
   0.5049370527267456,
   0.5990331172943115,
   0.7860387563705444,
   0.6323204040527344,
   0.5237816572189331,
   0.5449946522712708,
   0.6082255244255066,
   0.7562646865844727,
   0.7559481263160706,
   0.586929202079773,
   0.8536381125450134,
   0.8005446791648865,
   0.4814322590827942,
   0.39397647976875305,
   0.4920624792575836,
   0.40677472949028015,
   0.4531638026237488,
   0.5364774465560913,
   0.48207107186317444